# Workbook to CIRRUS and GaP LGR

This notebook showcases the complete workflow from a spreadsheet well description to a CIRRUS-ready LGR case. Change `case_name` in the configuration cell to explore either the Wildcat or Smeaheia example.

The notebook stages files and inspects the generated inputs without launching CIRRUS automatically. The simulator run is explicit in the final optional cell.

## Workflow

```text
XLSX -> well_input.json -> parameterized TEMP-0.in
     -> CIRRUS initialization -> .EGRID + .INIT
     -> GaP LGRBuilder -> TEMP_LGR.grdecl
     -> final CIRRUS simulation
```

The same workbook supplies physical well data and scenario/grid assumptions. The Survey sheet can remain empty for a vertical well or contain measured-depth deviation data.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'test_data').exists():
    repo_root = repo_root.parent

case_name = 'wildcat'  # Change to 'smeaheia' to explore the other example.
case_root = repo_root / 'test_data' / 'examples' / case_name
workbook = case_root / f'{case_name}_workbook.xlsx'
output_root = repo_root / 'notebook-output' / f'{case_name}-case'

assert case_name in {'wildcat', 'smeaheia'}
assert workbook.exists(), f'Missing workbook: {workbook}'
print(f'case: {case_name}')
print(f'workbook: {workbook}')
print(f'output: {output_root}')

In [ ]:
import pandas as pd

workbook_sheets = pd.ExcelFile(workbook, engine='openpyxl').sheet_names
print('sheets:', ', '.join(workbook_sheets))

grid_policy = pd.read_excel(workbook, sheet_name='GridPolicy', engine='openpyxl')
assumptions = pd.read_excel(workbook, sheet_name='SubsurfaceAssumptions', engine='openpyxl')
survey = pd.read_excel(workbook, sheet_name='Survey', engine='openpyxl')

display(grid_policy)
display(assumptions)
display(survey)

In [ ]:
stage_command = [
    sys.executable,
    str(repo_root / 'runscripts' / 'prepare_init_case_from_xlsx.py'),
    '--xlsx', str(workbook),
    '--output-root', str(output_root),
    '--write-well-json',
    '--force',
]
result = subprocess.run(stage_command, cwd=repo_root, check=True, capture_output=True, text=True)
print(result.stdout)

In [ ]:
for path in sorted(output_root.rglob('*')):
    if path.is_file():
        print(path.relative_to(output_root))

well_json = output_root / 'well_input.json'
deck = output_root / 'model' / 'TEMP-0.in'
grdecl = output_root / 'include' / 'TEMP_GRD.grdecl'

well_payload = json.loads(well_json.read_text(encoding='utf-8'))
print('well identifier:', well_payload['spec']['well_header']['unique_wellbore_identifier'])
print('deck dates and equilibration:')
for line in deck.read_text(encoding='utf-8').splitlines():
    if any(key in line for key in ('START_DATE', 'FINAL_DATE', 'DATUM_D', 'PRESSURE', 'WGC_D')):
        print(line)
print('initialization LGR include present:', 'TEMP_LGR.grdecl' in grdecl.read_text(encoding='utf-8'))

## Run CIRRUS initialization

The staged deck is initialization-only by default. On a Linux host with CIRRUS installed, set the command in the next cell and run it. This must produce both `.EGRID` and `.INIT` before the GaP step.

In [ ]:
# Set this to your site-specific CIRRUS command before running.
cirrus_command = None  # Example: 'runcirrus -i -nm 6 {deck}'

if cirrus_command is None:
    print('Skipped: set cirrus_command to run CIRRUS initialization.')
else:
    command = cirrus_command.format(deck=str(deck.resolve()))
    subprocess.run(command, cwd=deck.parent, shell=True, check=True)
    egrid = output_root / 'model' / 'TEMP-0.EGRID'
    init = output_root / 'model' / 'TEMP-0.INIT'
    assert egrid.exists() and init.exists(), 'CIRRUS did not produce EGRID and INIT'
    print('CIRRUS initialization complete')

## Build the GaP LGR

After CIRRUS initialization has produced `.EGRID` and `.INIT`, run the following cell. It uses the same selected workbook-derived `well_input.json`.

In [ ]:
sim_case = output_root / 'model' / 'TEMP-0'
lgr_command = [
    sys.executable,
    str(repo_root / 'runscripts' / 'build_lgr_from_json.py'),
    '--well-json', str(well_json),
    '--sim-case', str(sim_case),
    '--output-folder', str(output_root / 'include'),
    '--lgr-name', 'TEMP_LGR',
]
if not sim_case.with_suffix('.EGRID').exists() or not sim_case.with_suffix('.INIT').exists():
    print('Skipped: run CIRRUS initialization first.')
else:
    result = subprocess.run(lgr_command, cwd=repo_root, check=True, capture_output=True, text=True)
    print(result.stdout)

In [ ]:
# Configure the same deck for the final run after TEMP_LGR.grdecl exists.
final_command = [
    sys.executable,
    str(repo_root / 'runscripts' / 'prepare_init_case_from_xlsx.py'),
    '--xlsx', str(workbook),
    '--output-root', str(output_root),
    '--final-run',
    '--force',
]
if not (output_root / 'include' / 'TEMP_LGR.grdecl').exists():
    print('Skipped: build TEMP_LGR.grdecl first.')
else:
    result = subprocess.run(final_command, cwd=repo_root, check=True, capture_output=True, text=True)
    print(result.stdout)
    print('The shared deck is configured for the final CIRRUS simulation.')

## Load GRDECL data

Load the selected case's generated GRDECL text and extract the key grid declarations for inspection.

In [ ]:
import re

grid_text = grdecl.read_text(encoding='utf-8')
dimens_match = re.search(r'DIMENS\s+(\d+)\s+(\d+)\s+(\d+)\s*/', grid_text)
assert dimens_match, 'GRDECL is missing DIMENS'
nx, ny, nz = map(int, dimens_match.groups())
print(f'grid dimensions: nx={nx}, ny={ny}, nz={nz}')
print(grid_text[:500])

## Validate grid and property consistency

Check that the staged GRDECL contains the expected region and property assignments for the selected case.

In [ ]:
required_files = [
    workbook,
    well_json,
    deck,
    grdecl,
    output_root / 'include' / 'tops_dz.inc',
    output_root / 'include' / 'co2_db_new.dat',
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing selected-case files: {missing}')

assert nx > 0 and ny > 0 and nz > 0
assert 'EQLNUM 1' in grid_text and 'EQLNUM 2' in grid_text
assert 'FIPLEG' in grid_text and 'PERMX' in grid_text
print('grid and required artifacts are consistent')

## Run shared processing pipeline

The same summary logic is applied to either workbook so the selected case is the only changing input.

In [ ]:
from src.WellClass.libs.well_class import WellProcessed

processed_well = WellProcessed.from_json(well_json)
summary = {
    'case': case_name,
    'well_identifier': processed_well.header['unique_wellbore_identifier'],
    'construction_intervals': len(processed_well.hole_casings or []),
    'borehole_intervals': len(processed_well.borehole or []),
    'plug_intervals': len(processed_well.processed_plugs or []),
    'survey_points': len(processed_well.survey.get('md_rkb', [])) if processed_well.survey else 0,
}
summary

## Visualize selected-case outputs

Use a compact summary plot to compare the selected well and grid setup.

In [ ]:
import matplotlib.pyplot as plt

labels = ['construction', 'borehole', 'plugs', 'survey']
values = [summary['construction_intervals'], summary['borehole_intervals'], summary['plug_intervals'], summary['survey_points']]
fig, axis = plt.subplots(figsize=(7, 3.5))
axis.bar(labels, values, color=['#168aad', '#52b788', '#f4a261', '#e76f51'])
axis.set_title(f'{case_name.title()} input summary')
axis.set_ylabel('count')
axis.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

## Export results and artifacts

Save the selected-case summary separately so Wildcat and Smeaheia runs remain reproducible and do not overwrite one another.

In [ ]:
artifacts = {
    **summary,
    'workbook': str(workbook),
    'well_json': str(well_json),
    'deck': str(deck),
    'grdecl': str(grdecl),
}
summary_path = output_root / 'notebook_summary.json'
summary_path.write_text(json.dumps(artifacts, indent=2), encoding='utf-8')
print(f'saved summary: {summary_path}')